# astrophoto
- une mini-app pour réduire, additionner, et améliorer des astrophoto
- réduction des bruts à partir de dark et flat
- addition des bruts
- création de 3 masques : étoiles, fond de ciel et nébuleuses/galaxies
- traitement spécifique aux 3 couches
- retrait du gradient si besoin
- ajustement de la balance des blancs
- réhaussement de la clarté
- réhaussement des couleurs

**attention** : n'accepte que des fichiers de type FITS ou PNG (n&b ou couleur)

Pour des fichiers RAW, il faut d'abord les convertire par exemple avec *imagemagick* :

-> Exemple de création des masters, avec binning 2 (50%)
```
$ magick mogrify -format png -depth 16 -define dng:use-auto-bright=false -define dng:use-camera-wb=true -define dng:no-auto-bright=true -gamma 1.0 -filter box -resize 50% flat_00*.CR2
$ magick flat_00*.png -evaluate-sequence median masterflat.png
$ magick mogrify -format png -depth 16 -define dng:use-auto-bright=false -define dng:use-camera-wb=true -define dng:no-auto-bright=true -gamma 1.0 -filter box -resize 50% dark_00*.CR2
$ magick dark_00*.png -evaluate-sequence median masterdark.png
```
-> Exemple de création des bruts, avec binning 2 (50%)
```
$ magick mogrify -format png -depth 16  -define dng:use-auto-bright=false -define dng:use-camera-wb=true -define dng:no-auto-bright=true -gamma 1.0 -filter box -resize 50% m42_00*.CR2
```

ou *dcraw* : 

```

$ ....

```




In [1]:
%matplotlib widget

import os, glob, time, functools
import time
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import rgb_to_hsv, hsv_to_rgb 
from astropy.io import fits
from astropy.stats import sigma_clipped_stats
from astropy.visualization import AsinhStretch
from photutils.segmentation import detect_threshold, detect_sources
from scipy import ndimage
from scipy.fft import fft2, ifft2
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output
from ipyfilechooser import FileChooser

# --- CONFIGURATION ---
plt.close('all')
plt.ioff()
plt.style.use('dark_background')

class PhotoLab:
    def __init__(self, logger_callback=None):
        self.calibrated = None
        self.masks = {}
        self.log = logger_callback if logger_callback else print

    def _load_single_raw(self, path):
        ext = os.path.splitext(path)[1].lower()
        if ext in ['.fit', '.fits', '.fts']:
            with fits.open(path) as h: 
                return h[0].data.astype(np.float32)
                
        return np.array(Image.open(path)).astype(np.float32)

    def _get_shift(self, img1, img2):
        r = np.mean(ref, axis=2) if ref.ndim == 3 else ref
        t = np.mean(target, axis=2) if target.ndim == 3 else target
        prod = fft2(r) * fft2(t).conj()
        cc = ifft2(prod / (np.abs(prod) + 1e-10))
        shift = np.unravel_index(np.argmax(np.abs(cc)), r.shape)
        return [s if s <= r.shape[i]//2 else s - r.shape[i] for i, s in enumerate(shift)]

    def get_shift_fft(self, ref, target):
        i1 = np.log1p(img1)
        i2 = np.log1p(img2)
        # On force la moyenne à 0 et l'écart-type à 1
        i1 = (i1 - np.mean(i1)) / np.std(i1)
        i2 = (i2 - np.mean(i2)) / np.std(i2)
        return self._get_shift_fft(i1, i2)
        
    def remove_gradient(self, img):
        self.log(f"Removing gradient...")

        try:
            h, w = img.shape[:2]
            out = img.copy()
            
            # Paramètres de la grille (8x8 zones pour M31)
            grid_size = 8
            step_y, step_x = h // grid_size, w // grid_size
            
            for i in range(img.shape[2]):
                chan = img[:,:,i]
                # Création d'une carte de fond basse résolution
                bg_map_small = np.zeros((grid_size, grid_size))
                
                for gy in range(grid_size):
                    for gx in range(grid_size):
                        # Extraction d'une zone (tile)
                        tile = chan[gy*step_y:(gy+1)*step_y, gx*step_x:(gx+1)*step_x]
                        # On prend le 10ème percentile pour éviter de mesurer la galaxie ou les étoiles
                        bg_map_small[gy, gx] = np.percentile(tile, 10)
                
                # On ré-étire cette carte à la taille d'origine avec un lissage cubique
                # pour créer un modèle de gradient très doux
                bg_model = ndimage.zoom(bg_map_small, (h/grid_size, w/grid_size), order=3)
                bg_model = bg_model[:h, :w] # Recalage dimensions
                
                # Soustraction souple
                out[:,:,i] = np.clip(chan - bg_model, 0, 1)
            
            # Rééquilibrage final pour éviter les zones "trouées"
            return out - np.min(out)
        except Exception as e:
            return img
            
    def load_files(self, file_paths):
        if not file_paths: return
            
        self.dark, self.flat = 0, 1

        # charge les master s'ils existent
        for f in glob.glob("master*"):
            try:
                data = self._load_single_raw(f)
                norm = data / (65535.0 if data.max() > 255 else 255.0)
                if "dark" in f.lower(): 
                    self.dark = norm
                    self.log(f"masterdark loaded")
                if "flat" in f.lower(): 
                    self.flat = norm / (np.mean(norm) + 1e-10)
                    self.log(f"masterflat loaded")

            except: pass

        all_aligned = []
        ref_data, target_shape = None, None
        for i, p in enumerate(sorted(file_paths)):
            if "master" in os.path.basename(p).lower(): continue
                
            try:
                raw = self._load_single_raw(p) / (65535.0 if self._load_single_raw(p).max() > 255 else 255.0)
                self.log(f"Reducing image {p}...")
                calib = (raw - self.dark) / (self.flat + 1e-6)

                if target_shape is None: 
                    target_shape, ref_data = calib.shape, calib
                    all_aligned.append(calib)
                else:
                    if calib.shape != target_shape:
                        if calib.shape[0] == target_shape[1]: calib = np.rot90(calib)
                        else: continue
                    shift = self.get_shift_fft(ref_data, calib)
                    #shift = get_shift_fft(np.log1p(ref_data), np.log1p(calib))
                    #print(shift)
                    aligned = np.stack([ndimage.shift(calib[:,:,c], shift, order=1) for c in range(3)], axis=-1) if calib.ndim == 3 else ndimage.shift(calib, shift, order=1)
                    all_aligned.append(aligned)
                self.log(f"Aligning image {i+1}/{len(file_paths)}...")
            except: pass
            
        if all_aligned:
            stacked = np.mean(all_aligned, axis=0)
            if stacked.ndim == 3:
                for c in range(3): stacked[:,:,c] -= np.percentile(stacked[:,:,c], 25)
            p_high = np.percentile(stacked, 99.9)
            self.calibrated = np.clip(stacked / (p_high + 1e-10), 0, 1)
            if self.calibrated.ndim == 2: self.calibrated = self.calibrated[:,:,None]
            self.log(f"Load and stacking complete ({len(all_aligned)} images)")

    def extract_masks(self, img, stars_sigma=2, galaxy_sigma=15):
        # 1. Image en gris pour l'analyse
        gray = np.mean(img, axis=2)
        
        # 2. Masque des Étoiles (toujours actif)
        seuil_etoile = np.mean(gray) + (2.0 * np.std(gray))
        s_mask = (gray > seuil_etoile).astype(float)
        s_mask = ndimage.grey_dilation(s_mask, size=(3,3))
        s_mask = ndimage.gaussian_filter(s_mask, sigma=stars_sigma)
        
        # 3. Masque de la Galaxie AVEC CONDITION "OFF"
        if galaxy_sigma <= 0.1:
            # SI SIGMA = 0 : On crée un masque tout blanc (1.0 partout)
            # L'image entière est considérée comme une galaxie
            g_mask = np.ones_like(gray)
        else:
            # SINON : On calcule le masque normalement
            gray_smooth = ndimage.gaussian_filter(gray, sigma=3)
            p_gal = np.percentile(gray_smooth, 85.0)
            g_mask = (gray_smooth > p_gal).astype(float)
            g_mask = ndimage.gaussian_filter(g_mask, sigma=galaxy_sigma)
        
        # 4. Mise à jour du dictionnaire
        # Le fond (background) devient automatiquement 0 partout si g_mask est à 1
        self.masks = {
            'stars': np.clip(s_mask, 0, 1),
            'galaxy': np.clip(g_mask, 0, 1),
            'background': np.clip(1.0 - g_mask - s_mask, 0, 1)
        }
              
    def apply_cosmetics(self, stretch, bp_ratio, clarity, denoise, saturation, galaxy_sigma, stars_sigma, rgb=(1,1,1), do_grad=True):
        # 1. Préparation et recalcul des masques en direct
        img = self.calibrated.copy()
        
        # On force la mise à jour des masques avec les valeurs des curseurs
        # C'est ce qui permet au sigma=0 de désactiver le masque
        self.extract_masks(img, stars_sigma=stars_sigma, galaxy_sigma=galaxy_sigma)
        
        # Récupération des masques (avec la dimension 3D pour numpy)
        m_gal = self.masks['galaxy'][..., None]
        m_back = self.masks['background'][..., None]
        m_stars = self.masks['stars'][..., None]
    
        # 2. Prétraitement (Gradient et Balance des blancs)
        if do_grad:
            img = self.remove_gradient(img)
            
        for i in range(3):
            img[:,:,i] *= rgb[i]
    
        # 3. Réduction du bruit (Denoise)
        if denoise > 0:
            img_lisse = ndimage.gaussian_filter(img, sigma=denoise)
            if galaxy_sigma > 0.1:
                # Mode ciblé : On ne lisse que le fond
                img = (img_lisse * m_back) + (img * (1.0 - m_back))
            else:
                # Mode Global (sigma=0) : On lisse toute l'image
                img = img_lisse
    
        # 4. Accentuation (Clarity)
        if clarity > 0:
            # Calcul des détails fins
            fine = (img - ndimage.gaussian_filter(img, sigma=2)) * clarity
            # S'applique partout si galaxy_sigma=0 (car m_gal sera blanc partout)
            img = img + (fine * m_gal)
        
        # 5. Point Noir (Black Point)
        p25 = np.percentile(img, 25)
        p50 = np.percentile(img, 50)
        # Formule pour décoller le signal du bruit de fond
        blk = p25 + ((p50 - p25) * (bp_ratio - 1.0) * 5)
        img_lin = np.clip(img - blk, 0, 1)
        
        # 6. Saturation des couleurs
        if saturation != 1.0:
            luma = np.mean(img_lin, axis=2, keepdims=True)
            # Éloigne les canaux RVB de la luminance moyenne
            img_lin = np.clip(luma + (img_lin - luma) * saturation, 0, 1)
    
        # 7. Stretch Final (Asinh)
        # Préserve les couleurs dans les hautes lumières contrairement au Log
        img_finale = np.clip(AsinhStretch(a=stretch)(img_lin), 0, 1)
        
        return img_finale
    

class AstroDashboard:
    def __init__(self, lab):
        self.lab = lab
        self.output_plot = widgets.Output()
        self.output_hist = widgets.Output()
        
        self.status_dot = widgets.HTML(value="<div style='width: 12px; height: 12px; border-radius: 50%; background-color: #2ecc71; margin-right: 8px;'></div>")
        self.log_text = widgets.Label(value="Ready")
        self.log_box = widgets.HBox([self.status_dot, self.log_text], layout={'align_items': 'center', 'margin': '0 0 5px 10px'})
        self.lab.log = lambda m: setattr(self.log_text, 'value', f"{m}")

        with self.output_plot:
            self.fig, self.ax = plt.subplots(figsize=(9, 7), facecolor='black')
            plt.subplots_adjust(0, 0, 1, 1)
            self.fig.canvas.header_visible = False
            self.fig.canvas.toolbar_position = 'right'
            self.fig.canvas.layout.width, self.fig.canvas.layout.height = '100%', '100%'
            self.fig.canvas.layout.min_height = '500px'
            display(self.fig.canvas)

        with self.output_hist:
            self.fig_h, self.ax_h = plt.subplots(figsize=(3, 1), facecolor='black')
            plt.subplots_adjust(0, 0, 1, 1); self.fig_h.canvas.header_visible = False
            self.fig_h.canvas.toolbar_visible = False; display(self.fig_h.canvas)

        self.fc = FileChooser(os.getcwd(), title='<b>Select a directory : </b>', dir_only=True, show_only_dirs=True)
        self.fl = widgets.SelectMultiple(options=[], layout={'height': '120px', 'width': '100%'})
        self.btn_load = widgets.Button(description="Load", button_style='info', layout={'width':'49%'})
        self.btn_exp = widgets.Button(description="Export", button_style='success', layout={'width':'49%'})
        self.btn_mask_toggle = widgets.ToggleButton(value=False, description='Show Masks', button_style='warning', icon='eye', layout={'width':'100%'})
        self.chk_grad = widgets.Checkbox(value=False, description='Gradient removal', indent=False)
        
        s_sty, s_lay = {'description_width': '100px'}, {'width': '100%'}
        self.sliders = {
            'Stars σ': widgets.FloatSlider(value=15, min=0.01, max=20, step=0.5, description='Stars σ', style=s_sty, layout=s_lay, continuous_update=False),
            'Galaxy σ': widgets.FloatSlider(value=20, min=0, max=80, step=0.1, description='Galaxy σ', style=s_sty, layout=s_lay, continuous_update=False),
            'Stretch': widgets.FloatSlider(value=0.02, min=0.001, max=0.1, step=0.001, description='Stretch', style=s_sty, layout=s_lay, continuous_update=False),
            'BlackPt': widgets.FloatSlider(value=1.0, min=0, max=2, step=0.01, description='BlackPt', style=s_sty, layout=s_lay, continuous_update=False),
            'Clarity': widgets.FloatSlider(value=0.0, min=0, max=3, step=0.1, description='Clarity', style=s_sty, layout=s_lay, continuous_update=False),
            'Denoise': widgets.FloatSlider(value=0.0, min=0, max=5, step=0.1, description='Denoise', style=s_sty, layout=s_lay, continuous_update=False),
            'Red': widgets.FloatSlider(value=1.0, min=0.5, max=2.0, step=0.05, description='Red', style=s_sty | {'handle_color': 'red'}, layout=s_lay, continuous_update=False),
            'Green': widgets.FloatSlider(value=1.0, min=0.5, max=2.0, step=0.05, description='Green', style=s_sty | {'handle_color': 'green'}, layout=s_lay, continuous_update=False),
            'Blue': widgets.FloatSlider(value=1.0, min=0.5, max=2.0, step=0.05, description='Blue', style=s_sty | {'handle_color': 'blue'}, layout=s_lay, continuous_update=False),
            'Saturation': widgets.FloatSlider(value=1.0, min=1.0, max=3.0, step=0.1, description='Saturation',style=s_sty, layout=s_lay, continuous_update=False),
        }

        self.fc.register_callback(self._update_list)
        self.btn_load.on_click(self._on_load_click)
        self.btn_exp.on_click(lambda b: self._export())
        self.btn_mask_toggle.observe(lambda x: self._refresh(False, f"Show Mask status : {'Masques' if x['new'] else 'Image'}"), 'value')
        self.chk_grad.observe(lambda x: self._refresh(False, f"Gradient removal status : {'ON' if x['new'] else 'OFF'}"), 'value')
        
        for name, s in self.sliders.items():
            is_mask = name in ['Stars σ', 'Galaxy σ']
            s.observe(lambda x, n=name, m=is_mask: self._refresh(m, f"Settings change : {n}"), 'value')

    # --- DECORATEUR DE STATUT ---
    def busy_indicator(func):
        @functools.wraps(func)
        def wrapper(self, *args, **kwargs):
            self._set_status(True)
            try:
                return func(self, *args, **kwargs)
            finally:
                self._set_status(False)
        return wrapper

    def _set_status(self, busy=True):
        color = "#e74c3c" if busy else "#2ecc71"
        self.status_dot.value = f"<div style='width: 12px; height: 12px; border-radius: 50%; background-color: {color}; margin-right: 8px;'></div>"

    def _update_list(self, c):
        path = c.selected_path
        f = sorted(glob.glob(os.path.join(path, "*.fit*")) + glob.glob(os.path.join(path, "*.png")), key=os.path.getmtime, reverse=True)
        self.fl.options = [(os.path.basename(x), x) for x in f]

    @busy_indicator
    def _on_load_click(self, b):
        self.lab.log("Load in progress...")
        # On force un petit temps mort pour laisser le thread UI passer au rouge
        time.sleep(0.05) 
        self.lab.load_files(self.fl.value)
        self._refresh(True, "Display image...")

    @busy_indicator
    def _refresh(self, update_masks=False, log_msg=None):
        if self.lab.calibrated is None: return
        if log_msg: self.lab.log(log_msg)
        
        xlim, ylim = self.ax.get_xlim(), self.ax.get_ylim()
        has_zoom = xlim != (0.0, 1.0) and xlim != (0, 1)
        
        if update_masks: 
            self.lab.extract_masks(self.lab.calibrated, 
                           self.sliders['Stars σ'].value, 
                           self.sliders['Galaxy σ'].value)
            
        self.ax.clear()

        if self.btn_mask_toggle.value and self.lab.masks:
            self.lab.log(f"Render masks...")
            m = np.zeros((*self.lab.calibrated.shape[:2], 3))
            
            # 1. Étoiles -> Toujours en Rouge
            m += self.lab.masks['stars'][:,:,None] * [1, 0, 0]
            
            # 2. Galaxie -> Vert UNIQUEMENT si le masque est actif (sigma > 0)
            # Si galaxy_sigma est à 0, on ne rajoute pas de vert.
            g_sigma = self.sliders['Galaxy σ'].value
            if g_sigma > 0.1:
                m += self.lab.masks['galaxy'][:,:,None] * [0, 1, 0]
            
            # 3. Fond -> Bleu nuit (on peut aussi le masquer si sigma est à 0)
            if g_sigma > 0.1:
                m += self.lab.masks['background'][:,:,None] * [0, 0, 0.2]
            else:
                # En mode "Amas", on peut laisser le fond noir ou gris très sombre
                # pour bien voir les étoiles rouges.
                pass 
            
            self.ax.imshow(m)
            
        else:
            # Traitement normal de l'image
            self.lab.log(f"Render image...")
            rgb = (self.sliders['Red'].value, self.sliders['Green'].value, self.sliders['Blue'].value)
            
            img = self.lab.apply_cosmetics(
                    stretch=self.sliders['Stretch'].value,     
                    bp_ratio=self.sliders['BlackPt'].value,     
                    clarity=self.sliders['Clarity'].value,     
                    denoise=self.sliders['Denoise'].value,     
                    saturation=self.sliders['Saturation'].value,
                    galaxy_sigma=self.sliders['Galaxy σ'].value,  # On lie le curseur 'Galaxy σ'
                    stars_sigma=self.sliders['Stars σ'].value,   # On lie le curseur 'Stars σ'
                    rgb=rgb, 
                    do_grad=self.chk_grad.value
                )        
            # Affichage de l'image traitée
            # np.squeeze permet d'enlever les dimensions inutiles pour imshow
            self.ax.imshow(np.squeeze(img), cmap='gray' if img.ndim==2 or img.shape[2]==1 else None) 
        
        if has_zoom: self.ax.set_xlim(xlim); self.ax.set_ylim(ylim)
        self.ax.axis('off'); self.fig.canvas.draw_idle()

        # --- MISE À JOUR DE L'HISTOGRAMME ---
        self.ax_h.clear()
        
        # On étire les données calibrated pour l'affichage de l'histogramme
        # Cela permet de voir la "forme" du signal après stretch
        raw_stretch = AsinhStretch(a=self.sliders['Stretch'].value)(self.lab.calibrated)
        data = raw_stretch.ravel()
        
        # On filtre les pixels noirs ou saturés pour un histogramme plus lisible
        useful = data[(data > 0.0001) & (data < 0.9999)]
        
        if len(useful) > 0:
            # On définit les bords du graphique pour zoomer sur le signal utile
            h_min, h_max = np.percentile(useful, [0.5, 99.5])
            self.ax_h.hist(useful, bins=100, range=(h_min, h_max), color='cyan', alpha=0.4)
            
            # --- SYNCHRONISATION DU BLACK POINT ---
            # On reproduit exactement le calcul de apply_cosmetics
            # (Mais appliqué sur la version stretchée pour l'affichage graphique)
            p25 = np.percentile(raw_stretch, 25)
            p50 = np.percentile(raw_stretch, 50)
            std_fond = p50 - p25
            
            # Position de la ligne rouge (Black Point) sur l'histogramme
            cutoff = p25 + (std_fond * (self.sliders['BlackPt'].value - 1.0) * 5)
            
            self.ax_h.axvline(cutoff, color='red', linestyle='-', linewidth=2, alpha=0.8)
            
        self.ax_h.axis('off')
        self.fig_h.canvas.draw_idle()
        self.lab.log(f"Ready")

    @busy_indicator
    def _export(self):
        self.lab.log(f"Saving image...")
        rgb = (self.sliders['Red'].value, self.sliders['Green'].value, self.sliders['Blue'].value)
        
        img = self.lab.apply_cosmetics(
                stretch=self.sliders['Stretch'].value,     
                bp_ratio=self.sliders['BlackPt'].value,     
                clarity=self.sliders['Clarity'].value,     
                denoise=self.sliders['Denoise'].value,     
                saturation=self.sliders['Saturation'].value,
                galaxy_sigma=self.sliders['Galaxy σ'].value,  # On lie le curseur 'Galaxy σ'
                stars_sigma=self.sliders['Stars σ'].value,   # On lie le curseur 'Stars σ'
                rgb=rgb, 
                do_grad=self.chk_grad.value
            )
        
        p = os.path.join(self.fc.selected_path, f"photo_{int(time.time())}.png")
        Image.fromarray((img * 255).astype(np.uint8)).save(p); 
        self.lab.log(f"Export complete : {p}")

 
    @busy_indicator    
    def display(self):
        clear_output()
        rgb_box = widgets.VBox([widgets.HTML("<b style='color:white; font-size:11px;'>COLOR BALANCE</b>"), 
                        self.sliders['Red'], self.sliders['Green'], self.sliders['Blue']], 
                        layout={'border': '1px solid #444', 'padding': '5px', 'margin': '5px 0'})

        mask_box = widgets.VBox([widgets.HTML("<b style='color:white; font-size:11px;'>MASKS</b>"), 
                        self.sliders['Stars σ'], self.sliders['Galaxy σ']], 
                        layout={'border': '1px solid #444', 'padding': '5px', 'margin': '5px 0'})
        
        cosm_box = widgets.VBox([widgets.HTML("<b style='color:white; font-size:11px;'>COSMETICS</b>"), 
                        self.sliders['Stretch'], self.sliders['BlackPt'], 
                        self.sliders['Clarity'], self.sliders['Denoise'], self.sliders['Saturation']],
                        layout={'border': '1px solid #444', 'padding': '5px', 'margin': '5px 0'})

        left = widgets.VBox([
            self.fc, self.fl, widgets.HBox([self.btn_load, self.btn_exp]), 
            self.btn_mask_toggle, self.chk_grad, 
            widgets.HTML("<b style='color:cyan; font-size:11px;'>Histogram (Red=Cutoff)</b>"), 
            self.output_hist, rgb_box, mask_box, cosm_box
        ], layout={'width':'380px', 'min_width':'380px', 'padding':'15px'})
        right = widgets.VBox([self.log_box, self.output_plot], layout={'width':'100%'})
        display(widgets.HBox([left, right], layout={'width':'100%', 'background-color':'#000'}))
        


gui = AstroDashboard(PhotoLab()); gui.display()

